In [1]:
import os
import numpy as np
from tqdm import tqdm
from PIL import Image

import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

2026-05-01 13:45:36.423350: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777643136.687628      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777643136.762690      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777643137.437010      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777643137.437074      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777643137.437081      57 computation_placer.cc:177] computation placer alr

In [9]:
BASE = "/kaggle/input/datasets/mnhlfuch/vqa-v2/VQA v2/VQA v2"

TRAIN_IMG = BASE + "/Images/train2014"
VAL_IMG   = BASE + "/Images/val2014"
TEST_IMG  = BASE + "/Images/test2015"

# previous notebook output
DATA_DIR = "/kaggle/input/notebooks/urmilvisariya/vqa-project-notebook-1/preprocessed"

train_data = np.load(DATA_DIR + "/train.npy", allow_pickle=True)
val_data   = np.load(DATA_DIR + "/val.npy", allow_pickle=True)
test_data  = np.load(DATA_DIR + "/test.npy", allow_pickle=True)

## Loading pre-trained ResNet50 CNN model

base_model.predict(img) will return a feature vector of (batch,7,7,2048) which after avg pooling becomes (batch,2048)

In [4]:
base_model = ResNet50(weights='imagenet', include_top=False, pooling='avg')

2026-05-01 14:07:42.918839: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


<ul>
    <li>rgb image = x*x*3</li>
    <li>resize = 224*224*3</li>
    <li>preproces_input = 224*224*3 -> Normalized pixel values (e.g., -127 to 127) (zero centering)</li>
    <li>expand_dims = 1*224*224*3 where 1 represent batch size</li>
</ul>


In [5]:
def load_image(image_id, folder):
    filename = f"COCO_{folder.split('/')[-1]}_{image_id:012d}.jpg"
    path = os.path.join(folder, filename)
    
    img = Image.open(path).convert("RGB")
    img = img.resize((224, 224))
    
    img = np.array(img)
    img = preprocess_input(img)
    
    return np.expand_dims(img, axis=0)

In [6]:
def extract_features(data, img_folder):
    features = {}

    for item in tqdm(data):
        image_id = item["image_id"]

        if image_id in features:
            continue

        img = load_image(image_id, img_folder)
        feat = base_model.predict(img, verbose=0)

        features[image_id] = feat.flatten()

    return features

In [10]:
# Extracting features for just 5000 training images and 2000 validation images
train_features = extract_features(train_data[:5000], TRAIN_IMG)
val_features   = extract_features(val_data[:2000], VAL_IMG)

100%|██████████| 2000/2000 [01:04<00:00, 31.06it/s] 


In [22]:
# train_features

In [21]:
train_features[458752],len(train_features[458752],)

(array([1.1224722 , 1.0039166 , 0.18151991, ..., 2.5899515 , 1.2547791 ,
        0.        ], dtype=float32),
 2048)

In [23]:
# When you extract all 0.4 million image features, remember to store them in batches rather than saving all the features in a single .npy file.

In [11]:
np.save("/kaggle/working/train_features.npy", train_features)
np.save("/kaggle/working/val_features.npy", val_features)